## Check: new MCMC implementation vs. the original

`src/mcmc.py` contains two samplers:

* `adaptive_mcmc_spike_slab_original`: the original implementation (unchanged, kept for this comparison);
* `adaptive_mcmc_spike_slab`: the new implementation used for all results in the paper.

This notebook checks, on real data from the paper, that

1. the new likelihood and prior give **the same numbers** as the original functions;
2. the running covariance used by the new sampler equals `np.cov` of the chain;
3. both samplers give **the same posterior** (up to Monte Carlo error), and how long each takes;
4. why the original is slow: its run time grows **quadratically** with the number of iterations;
5. the saved results in `results/` are reproduced exactly by re-running the new sampler with the saved seed.

The two samplers use different random-number streams, so their chains are not identical draw by draw; they are
compared as samples from the same posterior.

The one deliberate change in the algorithm: the new sampler estimates the proposal covariance from the states after
iteration `adapt_from` (default `burn_in // 2`), the original from all states. Section 3 also runs the new sampler with
`adapt_from=0`, i.e. exactly the original adaptation.

**Run time:** about 10 minutes, almost all of it the original sampler.

In [1]:
import sys, os, time, io, contextlib
import numpy as np, pandas as pd
sys.path.append(os.path.abspath(".."))
with contextlib.redirect_stdout(io.StringIO()):
    from src.parsing import *
    from src.inference import *
    from src.mcmc import *
    import src.mcmc as M

def load_channel(network_file, traj_file, index):
    """Local data of one stoichiometric channel, exactly as in the example notebooks."""
    with contextlib.redirect_stdout(io.StringIO()):
        (_, _, _, _, trueTheta, _, _, reactant_matrix, unique_changes, compatible_reactions, _) = load_reaction_network(network_file)
        tl, sl = load_trajectory(traj_file)
        _, jc, wt, pr = parse_trajectory(sl, tl, reactant_matrix, unique_changes, compatible_reactions, verbose=False)
        lc, lw, lp, sel = extract_local_data(jc, wt, pr, unique_changes, index=index, deltaX=unique_changes[index], verbose=False)
    return lc, lw, lp, trueTheta[compatible_reactions[sel]], trueTheta

# Ergodic example (antithetic integral controller), T = 500, channel (+1,0,0): 4 candidate reactions, 2 active
lc, lw, lp, localTheta, trueTheta = load_channel("../data/example6_network.json", "../data/example6_T500_trajectory.json", 0)
print("visited states in this channel:", len(lp), "| events:", sum(lc.values()), "| true local rates:", localTheta)


visited states in this channel: 1537 | events: 1547 | true local rates: [2. 0. 1. 0.]


### 1. Same likelihood and prior

In [2]:
rng = np.random.default_rng(1)
G, n, tau = M._local_data_arrays(lc, lw, lp)
from scipy.special import gammaln
rows = []
for _ in range(1000):
    theta = rng.uniform(0, 3, len(localTheta)) * (rng.random(len(localTheta)) < 0.6)   # include exact zeros
    a = local_log_likelihood(lc, lw, lp, theta)               # original (src/inference.py)
    b = M._local_log_likelihood_arrays(G, n, tau, theta)      # new, vectorised
    ts = rng.normal(0, 3, len(localTheta))
    pa = log_prior_theta_star(ts, alpha=2, beta=0.25, pi=0.75, sigma_spike=10.0)   # original (scipy)
    pb = M._log_prior_theta_star_fast(ts, np.log(.75), np.log(.25), 2, .25, 10.0, 2*np.log(.25) - gammaln(2))
    rows.append((a, b, pa, pb))
r = np.array(rows)
fin = np.isfinite(r[:, 0])
print("likelihood: both -inf in the same cases:", np.array_equal(np.isfinite(r[:, 0]), np.isfinite(r[:, 1])))
print("likelihood: max relative difference    :", np.max(np.abs(r[fin, 0] - r[fin, 1]) / np.abs(r[fin, 0])))
print("prior     : max absolute difference    :", np.max(np.abs(r[:, 2] - r[:, 3])))


likelihood: both -inf in the same cases: True
likelihood: max relative difference    : 8.31598756853249e-13
prior     : max absolute difference    : 3.552713678800501e-15


### 2. Running covariance = `np.cov`

In [3]:
X = rng.normal(size=(5000, 4)) * [1, 10, 0.01, 3]
w_n, w_mean, w_m2 = 1, X[0].copy(), np.zeros((4, 4))
for k in range(1, len(X)):                 # the update used in adaptive_mcmc_spike_slab
    w_n += 1; d = X[k] - w_mean; w_mean += d / w_n; w_m2 += np.outer(d, X[k] - w_mean)
print("max |running covariance - np.cov| =", np.abs(w_m2 / (w_n - 1) - np.cov(X.T)).max())


max |running covariance - np.cov| = 2.842170943040401e-14


### 3. Same posterior, and run times

20,000 iterations, burn-in 4,000, thinning 10, three runs each. (The paper uses 500,000 iterations; the original is
far too slow for that here, see section 4.)

In [4]:
N, B, TH = 20_000, 4_000, 10
kw = dict(num_iterations=N, alpha=2, beta=0.25, pi=0.75, burn_in=B, adapt_every_n=10, printEveryNSteps=10**9)

def summarise(chain, label, seconds):
    k = np.asarray(chain)[B::TH]
    out = {"sampler": label, "seconds": round(seconds, 1)}
    for j in range(len(localTheta)):
        out[f"mean k{j}"] = round(k[:, j].mean(), 3)
        out[f"Pr(off) k{j}"] = round((k[:, j] <= 1e-3).mean(), 3)
    return out

rows = []
for s in range(3):
    init = np.random.default_rng([99, s]).uniform(0.001, 2, len(localTheta))
    np.random.seed(s); t = time.time()
    ch = adaptive_mcmc_spike_slab_original(lc, lw, lp, init.copy(), trueTheta, **kw)
    rows.append(summarise(ch, "original", time.time() - t))
    t = time.time()
    ch = adaptive_mcmc_spike_slab(lc, lw, lp, init.copy(), trueTheta, seed=s, adapt_from=0, **kw)
    rows.append(summarise(ch, "new, adapt_from=0 (original adaptation)", time.time() - t))
    t = time.time()
    ch = adaptive_mcmc_spike_slab(lc, lw, lp, init.copy(), trueTheta, seed=s, **kw)
    rows.append(summarise(ch, "new, default (adapt from burn_in/2)", time.time() - t))
res = pd.DataFrame(rows)
print("true rates:", localTheta)
print(res.to_string(index=False))
print("\nmean seconds per run:")
print(res.groupby("sampler")["seconds"].mean().round(1).to_string())


true rates: [2. 0. 1. 0.]
                                sampler  seconds  mean k0  Pr(off) k0  mean k1  Pr(off) k1  mean k2  Pr(off) k2  mean k3  Pr(off) k3
                               original     63.4    2.118         0.0      0.0       1.000    0.934         0.0      0.0         1.0
new, adapt_from=0 (original adaptation)      0.7    2.121         0.0      0.0       1.000    0.932         0.0      0.0         1.0
    new, default (adapt from burn_in/2)      0.7    2.116         0.0      0.0       1.000    0.935         0.0      0.0         1.0
                               original     62.1    2.116         0.0      0.0       1.000    0.938         0.0      0.0         1.0
new, adapt_from=0 (original adaptation)      0.7    2.114         0.0      0.0       1.000    0.938         0.0      0.0         1.0
    new, default (adapt from burn_in/2)      0.7    2.111         0.0      0.0       1.000    0.941         0.0      0.0         1.0
                               original    

### 4. Why the original is slow

Two parts, measured below:

* **per iteration:** the original evaluates the likelihood twice per iteration (proposal *and* the unchanged current
  state) with a Python loop over all visited states, and the prior twice with `scipy.stats`. A profile shows where the
  time goes. This cost is the same for every iteration.
* **per covariance update:** every 10 iterations after burn-in, the original converts the whole chain so far to an
  array and recomputes its covariance. This is cheap for short chains but grows with the chain length, and at
  500,000 iterations (45,000 updates on chains of 50,000–500,000 states) it adds a comparable amount of time.

The new sampler caches the current state's likelihood and prior, evaluates the likelihood on arrays, and updates the
covariance incrementally, so its cost per iteration is small and constant.

In [7]:
import cProfile, pstats
init = np.random.default_rng(7).uniform(0.001, 2, len(localTheta))
np.random.seed(0)
prof = cProfile.Profile(); prof.enable()
adaptive_mcmc_spike_slab_original(lc, lw, lp, init.copy(), trueTheta, **dict(kw, num_iterations=10_000, burn_in=2_000))
prof.disable()
st = pstats.Stats(prof); total = st.total_tt
def cum(name):
    return sum(v[3] for k, v in st.stats.items() if k[2] == name)
print(f"original, 10,000 iterations: {total:.0f} s in total")
print(f"  likelihood (local_log_likelihood): {cum('local_log_likelihood'):.0f} s")
print(f"  prior (log_prior_theta_star)     : {cum('log_prior_theta_star'):.0f} s")


original, 10,000 iterations: 38 s in total
  likelihood (local_log_likelihood): 33 s
  prior (log_prior_theta_star)     : 3 s


In [6]:
# cost of ONE covariance update of the original (np.array of the chain so far, then np.cov)
chain_list = [np.random.rand(len(localTheta)) for _ in range(500_000)]
rows = []
for n_ in [50_000, 250_000, 500_000]:
    t = time.time()
    for _ in range(5):
        np.cov(np.array(chain_list[:n_]).T)
    rows.append({"chain length": n_, "seconds per update": round((time.time() - t) / 5, 3)})
print(pd.DataFrame(rows).to_string(index=False))
upd = np.arange(50_010, 500_001, 10)                               # update iterations for the paper's settings
per = np.interp(upd, [r["chain length"] for r in rows], [r["seconds per update"] for r in rows])
per_it = total / 10_000
print(f"\nestimated original run time for the paper's settings (500,000 iterations) on this channel:")
print(f"  per-iteration part   : {per_it * 500_000 / 60:.0f} min")
print(f"  covariance updates   : {per.sum() / 60:.0f} min")
t = time.time(); adaptive_mcmc_spike_slab(lc, lw, lp, init.copy(), trueTheta, seed=0, **dict(kw, num_iterations=500_000, burn_in=50_000))
print(f"new sampler, same settings, measured: {time.time() - t:.0f} s")


 chain length  seconds per update
        50000               0.011
       250000               0.056
       500000               0.100

estimated original run time for the paper's settings (500,000 iterations) on this channel:
  per-iteration part   : 58 min
  covariance updates   : 44 min


new sampler, same settings, measured: 26 s


### 5. The saved results are reproduced by the new sampler

Re-runs one channel exactly as the example notebooks do (`run_channel_mcmc`, 500,000 iterations, the saved seed) and
compares the chain with the one saved in `results/`. Predator–prey, channel $(-1,+1)$ ($A+B \to 2B$, rate 0.01).

In [6]:
lc4, lw4, lp4, th4, true4 = load_channel("../data/example4_network.json", "../data/example4_T40_trajectory.json", 6)
t = time.time()
with contextlib.redirect_stdout(io.StringIO()):
    chain = run_channel_mcmc(lc4, lw4, lp4, th4, true4, results_dir="/tmp/verify_mcmc", index=6,
                             total_count=sum(lc4.values()), seed=4, make_plots=False)
print(f"500,000 iterations in {time.time() - t:.0f} s")
saved = load_channel_chain("../results/example4_T40", 6)
print("max |re-run chain - saved chain| =", np.abs(chain[::10] - saved["chain"]).max(), "(0 or rounding-level)")
s = pd.read_excel("../results/example4_T40/mcmc_summary.xlsx")
print(s[s.Run_Index == 6][["Param_Index", "True_Theta", "Mean", "CI_lower", "CI_upper", "Prob_Off", "ESS", "Acceptance_Rate"]].to_string(index=False))


/Users/suzannesindi/Documents/GitHub/BayesCRNInference/src/mcmc.py:65: RuntimeWarning: invalid value encountered in scalar divide
  z = (np.mean(a) - np.mean(b)) / np.sqrt(np.var(a)/len(a) + np.var(b)/len(b))
/Users/suzannesindi/miniforge3/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]


500,000 iterations in 57 s
max |re-run chain - saved chain| = 2.0990154059319366e-16 (0 or rounding-level)
 Param_Index  True_Theta     Mean  CI_lower  CI_upper  Prob_Off         ESS  Acceptance_Rate
           0        0.00 0.000000   0.00000  0.000000       1.0         NaN          0.08186
           1        0.00 0.000000   0.00000  0.000000       1.0         NaN          0.08186
           2        0.01 0.010071   0.00961  0.010533       0.0 4370.723234          0.08186
